In [1]:
import numpy as np

class NeuralNetwork:
    def __init__(self, layer_sizes_list, learning_rate = 0.001, epochs = 50, thresh = 0.00001, batch_size = 100, patience = 5, regularization_lambda = 0.001, want_debug = False):
        self.layer_sizes_list = layer_sizes_list
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.thresh = thresh
        self.batch_size = batch_size
        self.regularization_lambda = regularization_lambda
        self.want_debug = want_debug
        self.patience = patience
        self.num_of_layers = len(self.layer_sizes_list) - 1
        self.weights = []
        self.init_weights()
    
    # This function initializes the weights (including extra row for bias) list of np.arrays with the values from the layer_sizes_list.
    # The matrices are created in a way that the multiplication of 2 following ones will be valid (num of cols in 1 = num of rows in 2)
    def init_weights(self):
        np.random.seed(42) # Make sure the random is the same between runs
        
        for i in range(self.num_of_layers):
            cur_layer_size = self.layer_sizes_list[i]
            next_layer_size = self.layer_sizes_list[i + 1]
            in_size = cur_layer_size + 1 # Adding 1 to num of rows to account for added bias
            out_size = next_layer_size
            
            weight_matrix = np.random.randn(in_size, out_size) * np.sqrt(2 / in_size) # Recommended for use of ReLU
            self.weights.append(weight_matrix)
    
    # The activation function for all the neurons (except output layer), and its derivative
    def relu(self, z):
        return np.maximum(0, z)
    
    def relu_derivative(self, z):
        return (z > 0).astype(float)

    # Forward part of Back Propagation
    def forward(self, batch):
        X_list = [batch]
        S_list = []

        for weight_num in range(len(self.weights) -1): # For all layers except output
            prev_X = X_list[-1]
            prev_X_with_bias = np.hstack([prev_X, np.ones((prev_X.shape[0], 1))]) # Add column of 1's for biases
            S = (prev_X_with_bias @ self.weights[weight_num])
            S_list.append(S)
            X = self.relu(S)
            X_list.append(X)

        # Apply softmax to the last layer (output) instead of ReLU
        prev_X = X_list[-1]
        prev_X_with_bias = np.hstack([prev_X, np.ones((prev_X.shape[0], 1))]) # Add column of 1's for biases
        S = (prev_X_with_bias @ self.weights[-1])
        S_list.append(S)
        X = self.softmax(S)
        X_list.append(X)
        
        return X_list, S_list
        
    # Backward part of Back Propagation
    def backward(self, X_list, S_list, y_true):
        m = y_true.shape[0]
        grads = []

        # The initial delta from the output layer
        delta = X_list[-1] - y_true

        for layer_num in reversed(range(self.num_of_layers)):
            prev_X = X_list[layer_num]
            prev_X_with_bias = np.hstack([prev_X, np.ones((prev_X.shape[0], 1))]) # Add column of 1's for biases
            dW = (prev_X_with_bias.T @ delta) / m # Average gradient of the batch
            grads.insert(0, dW)

            if layer_num > 0:
                W_no_bias = self.weights[layer_num][:-1, :] # Remove the last row (bias weights)
                delta = (delta @ W_no_bias.T) * self.relu_derivative(S_list[layer_num - 1]) # Use the S_list for ReLU derivative in the current layer-1 (without last layer that uses softmax)

        return grads
    
    # This function fits the NN model by using Back Propagation
    def Fit(self, X_train, y_train, X_valid, y_valid):
        
        num_of_samples = X_train.shape[0]
        encoded_y = self.one_hot_encode(y_train)
        
        best_val_loss = float('inf')
        best_weights = None
        epochs_without_improvement = 0
        val_scores = []
        
        for epoch in range(self.epochs):
            # Shuffle the training set each epoch according to a random permutation of the rows (to randomize the batches each epoch)
            permutation = np.random.permutation(num_of_samples)
            X_train = X_train[permutation]
            y_train = y_train[permutation]
            encoded_y = encoded_y[permutation]
            
            # Batching
            for i in range(0, num_of_samples, self.batch_size):
                X_batch = X_train[i:i + self.batch_size]
                y_batch = encoded_y[i:i + self.batch_size]
                
                X_list, S_list = self.forward(X_batch)
                grads = self.backward(X_list, S_list, y_batch)
                self.update_weights(grads)
                
            val_loss = self.compute_loss(X_valid, y_valid) # Validation check for Early Stopping
            val_score = self.score(X_valid, y_valid) # Validation check for choosing hyperparameters
            
            val_scores.append(val_score)
            
            if self.want_debug:
                print(f"Epoch {epoch + 1}: Validation loss = {val_loss:.4f}, Validation score = {val_score:.4f}")

            if val_loss + self.thresh < best_val_loss: # If there was a significant improvement
                best_val_loss = val_loss
                best_weights = [w.copy() for w in self.weights]
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= self.patience:
                    if self.want_debug:
                        print(f"Early stopping on epoch {epoch + 1}")
                    break
        
        # Keep the best weights
        if best_weights is not None:
            self.weights = best_weights 
            
        if self.want_debug:
            print(f"Average Validation score = {np.mean(val_scores):.4f}, Best Validation score = {np.max(val_scores):.4f}")
    
    # Log loss for early stopping using validation
    def compute_loss(self, X, y_true):
        y_true_encoded = self.one_hot_encode(y_true)
        X_list, _ = self.forward(X)
        y_pred = X_list[-1]
        
        # Avoid getting log(0) with clipping
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        log_loss = -np.sum(y_true_encoded * np.log(y_pred)) / len(y_true)
        
        # L2 Regularization to help reduce/ prevent overfitting
        l2_sum = sum(np.sum(w[:-1] ** 2) for w in self.weights)  # Skip the last row of biases
        l2_loss = self.regularization_lambda * l2_sum
        
        return log_loss + l2_loss

    def one_hot_encode(self, y):
        num_of_classes = self.layer_sizes_list[-1]
        return np.eye(num_of_classes)[y] # select the appropriate one-hot vectors from the Identity matrix according to the class (row) numbers in y
    
    # This function is used to update the weights at the end of each batch (with added weight decay)
    def update_weights(self, grads):
        for i in range(len(self.weights)):
            reg = np.vstack([self.regularization_lambda * self.weights[i][:-1], np.zeros((1, self.weights[i].shape[1]))])
            self.weights[i] -= self.learning_rate * (grads[i] + reg)
    
    # This function predicts the classification number of a feature vector X
    def predict(self, X):
        output = self.forward(X)[0][-1]
        return np.argmax(output, axis = 1)
    
    def predict_proba(self, X):
        output = self.forward(X)[0][-1]  # The output from softmax
        return output
    
    # This function calculates (and returns) the NN's accuracy score between 0 and 1
    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)
    
    # This function calculates the softmax of a vector for the one-hot in the last layer
    def softmax(self, vector):
        shift_vector = vector - np.max(vector, axis=1, keepdims = True) # Recommended online to subtract max for numerical stability (per row)
        e = np.exp(shift_vector)
        return e / np.sum(e, axis = 1, keepdims = True)
    


# MNIST Main

In [2]:
import pandas as pd
import numpy as np

def main():
    # Load datasets
    df_train = pd.read_csv("MNIST-train.csv")
    df_test = pd.read_csv("MNIST-test.csv")
    
    # Shuffle their data
    df_train = df_train.sample(frac = 1, random_state = 42).reset_index(drop = True)
    df_test = df_test.sample(frac = 1, random_state = 42).reset_index(drop = True)
    
    # Extract features (X) and labels (y) for test and train
    X_train = df_train.iloc[:, :-1].values  # All columns except the label column (last one)
    y_train = df_train.iloc[:, -1].values  # Extract labels
    
    X_test = df_test.iloc[:, :-1].values  # All columns except the label column (last one)
    y_test = df_test.iloc[:, -1].values  # Extract labels
    
    # Split training data into training (80%) and validation (20%) sets
    split_idx = int(0.8 * len(X_train))
    X_train, X_valid = X_train[:split_idx], X_train[split_idx:]
    y_train, y_valid = y_train[:split_idx], y_train[split_idx:]
    
    # Train the Neural Network
    NN = NeuralNetwork(layer_sizes_list = [784, 512, 256, 10], 
                       learning_rate = 0.001, epochs = 100, thresh = 0.00001, 
                       batch_size = 100, patience = 5, regularization_lambda = 0.01, want_debug = False)
    NN.Fit(X_train, y_train, X_valid, y_valid)
    
    # Compute score on training and test sets
    train_score = NN.score(X_train, y_train)
    test_score = NN.score(X_test, y_test)

    # Print results
    print(f"Training score: {train_score:.4f}")
    print(f"Testing score: {test_score:.4f}")
    
if __name__ == "__main__":
    main()
    

Training score: 0.9999
Testing score: 0.9628


# MB Main

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

def main():
    # Load datasets
    df_train = pd.read_csv("MB_data_train.csv")
    
    # Shuffle the data
    df_train = df_train.sample(frac = 1, random_state = 42).reset_index(drop = True)
    
    # Extract features (X) and labels (y) for test and train
    X_train = df_train.iloc[:, 1:].values  # All columns except the label column (first one in this case)
    y_train = df_train.iloc[:, 0].values  # Extract labels
    
    # Scale the features (on your X_test please use only scaler.transform - without fit)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    
    # Change all labels containing "Fibro" to 0, and all other labels (there are only 2 total - so containing "Ctrl") to 1
    y_train = np.where(np.char.find(y_train.astype(str), 'Fibro') != -1, 0, 1)
    
    # Fit a random forest
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    importances = rf.feature_importances_
    indices = np.argsort(importances)[::-1]  # Sort in descending order
    top_k = 41  # The number of top features to keep

    top_indices = indices[:top_k]
    X_train = X_train[:, top_indices] # for all samples in X_train select only top_k features (do this also on your X_test)
    
    # Split training data into training (80%) and validation (20%) sets
    split_idx = int(0.8 * len(X_train))
    X_train, X_valid = X_train[:split_idx], X_train[split_idx:]
    y_train, y_valid = y_train[:split_idx], y_train[split_idx:]
    
    # Train the Neural Network
    NN = NeuralNetwork(layer_sizes_list=[X_train.shape[1], 64, 32, 2], 
                       learning_rate = 0.01, epochs = 200, thresh = 0.00001, 
                       batch_size = 10, patience = 5, regularization_lambda = 0.1, want_debug = False)
    NN.Fit(X_train, y_train, X_valid, y_valid)
    
    train_score = NN.score(X_train, y_train)
    
    # Print results
    print(f"Training score: {train_score:.4f}")
    
if __name__ == "__main__":
    main()
    

Training score: 0.9500
